In [1]:
import pyfeyngym as pfg

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


# Set up topology and generate improved seeds with maximum rank 4

In [2]:
IBP_file = "IBP_LI" 
trivial_sector_file = "trivialsector"
modulus = 2**31-1
m_vals = {'d': 23, 'm1': 3, 'm2': 5, 'm3': 17, 'm4': 23}

In [3]:
eq_templates = pfg.gen_eq_templates(IBP_file, m_vals)

In [4]:
# Number of IBP+LI oeprators
len(eq_templates)

18

In [5]:
trivial_sectors = pfg.get_trivial_sectors(trivial_sector_file, cut=[1,3,6,8], n_indices=11)

In [6]:
top_sector = (1,1,1,1, 1,1,1,1, 0,0,0)

Let's first generate the usual seeds with tensor rank up to 4, with "improved seeding".
Then we convolute these seeds with a line connecting the power of the last ISP from 0 to 10,
to generate a "strip" with a finite width

In [7]:
s_max, r_max, d_max = 4, 8, 0

In [8]:
starting_seeds = pfg.gen_all_seeds(top_sector, trivial_sectors, s_max, r_max, d_max)

In [9]:
improved_seed_param = 4
improved_seeds = [s for s in starting_seeds if
    (pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - improved_seed_param)) or
                  (s[3]<=0 and s[4]<=0 and s[6]<=0 and pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - improved_seed_param + 1))]
# The last line makes an exception for sector 167, exactly as in the Kira config file.

In [10]:
len(improved_seeds)

358

In [11]:
seed_op_eq_list, all_variables = pfg.gen_eqs(eq_templates, trivial_sectors, m_vals, improved_seeds)
equations = [a[-1] for a in seed_op_eq_list]
sorted_vars = pfg.sort_integrals_desc(all_variables)
test_integral = (1,1,1,1,1,1,1,1, -1,-1,-2) # a rank-4 integral to to be reduced to obtain the master list
solution = pfg.solve_eqs_modulo(equations, sorted_vars, modulus, needed_variables = {test_integral})

current row = 5001


Reduce a rank-4 integral to obtain the master list

In [12]:
reduced = solution[(1,1,1,1,1,1,1,1, -1,-1,-2)]
masters = [term[0] for term in reduced]
masters

[(1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -1, 0),
 (1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0),
 (1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, 0, 0),
 (1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0),
 (1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0),
 (1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0),
 (1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1),
 (1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -2, 0),
 (1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, 0, -1),
 (1, 1, 1, 1, 0, 1, 1, 1, -1, 0, 0),
 (1, 1, 1, 1, -1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, 0, -1),
 (1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, -1, 1, 1, 1, 1, 1, 1, 0, 0, 0)]

In [13]:
len(masters)

27

# Reduction of (1, ... , 1, 0, 0, -20)

We will use a strip from (0, 0, 0) to (0, 0, 16), convoluted with the "improved seeds" for low-rank integrals with rank cutoff 4

In [14]:
seed_set = set()
# strip: (0, 0, 0) to (0, 0, 16)
for raise_rank in range(17):
    for seed in improved_seeds:
        last_index = seed[-1] - raise_rank
        seed_set.add(seed[:-1] + (last_index,))

In [15]:
len(seed_set)

4054

In [16]:
# Sort by descending complexity
final_seeds = pfg.sort_integrals_desc(list(seed_set))

In [17]:
seed_op_eq_list, all_variables = pfg.gen_eqs(eq_templates, trivial_sectors, m_vals, final_seeds)
equations = [a[-1] for a in seed_op_eq_list]
sorted_vars = pfg.sort_integrals_desc(all_variables)
len(seed_op_eq_list)

72972

In [18]:
%%time
target_integral = (1,1,1,1,1,1,1,1, 0,0,-20)
solution = pfg.solve_eqs_modulo(equations, sorted_vars, modulus,
                               keep_on_rhs = masters, complete_pivoting = True,
                               needed_variables = [target_integral])

current row = 57601


CPU times: user 28.5 s, sys: 1.74 s, total: 30.2 s
Wall time: 30.3 s


Let's verify that the target integral is reduced to masters

In [19]:
reduced = solution[target_integral]
reduced

[[(1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0), 194230414],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, 0), -1224246206],
 [(1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0), 1203365232],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0), -1686496366],
 [(1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0), -1409749802],
 [(1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0), -893747121],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0), -1288020795],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, 0, 0), -2037407497],
 [(1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0), -1299576305],
 [(1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0), -1767066587],
 [(1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0), 590206919],
 [(1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0), 742597859],
 [(1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0), -1301876043],
 [(1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0), -242695613],
 [(1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0), -2094504100],
 [(1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0), -442038936],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1), 14127233],
 [(1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0), -1102136370],
 [(1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0), -1497790073],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -2, 

In [20]:
# The number of terms is the same as the number of masters, so the reduction is complete
len(reduced)

27

# Reduction of (1, ... , 1, -8, -6, -6)

We will use three strips joined in a zig-zag shape

In [21]:
seed_set = set()
# Strip 1: (0, 0, 0) to (5, 0, 0)
for raise_rank in range(6):
    for seed in improved_seeds:
        last_three_indices = (seed[-3]-raise_rank, seed[-2], seed[-1])
        seed_set.add(seed[:-3] + last_three_indices)
# Strip 2: (4-5, 1, 0) to (4-5, 6, 0), 
for raise_rank in range(7):
    for seed in improved_seeds:
        last_three_indices = (seed[-3] - 5, seed[-2] - raise_rank, seed[-1])
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 4, seed[-2] - raise_rank, seed[-1])
        seed_set.add(seed[:-3] + last_three_indices)
# Strip 3: (4-5, 4-6, 1) to (4-5, 4-6, 6)
for raise_rank in range(1, 7):
    for seed in improved_seeds:
        last_three_indices = (seed[-3] - 4, seed[-2] - 6, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 5, seed[-2] - 6, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 4, seed[-2] - 5, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 5, seed[-2] - 5, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 4, seed[-2] - 4, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)
        last_three_indices = (seed[-3] - 5, seed[-2] - 4, seed[-1] - raise_rank)
        seed_set.add(seed[:-3] + last_three_indices)

In [22]:
len(seed_set)

8629

In [23]:
# Sort by descending complexity
final_seeds = pfg.sort_integrals_desc(list(seed_set))

In [24]:
seed_op_eq_list, all_variables = pfg.gen_eqs(eq_templates, trivial_sectors, m_vals, final_seeds)
equations = [a[-1] for a in seed_op_eq_list]
sorted_vars = pfg.sort_integrals_desc(all_variables)
len(seed_op_eq_list)

155322

In [25]:
%%time
target_integral = (1,1,1,1,1,1,1,1, -8,-6,-6)
solution = pfg.solve_eqs_modulo(equations, sorted_vars, modulus,
                               keep_on_rhs = masters, complete_pivoting = True,
                               needed_variables = {target_integral})

current row = 122701


CPU times: user 2min 17s, sys: 5.01 s, total: 2min 22s
Wall time: 2min 22s


Let's verify that the target integral is reduced to masters

In [26]:
reduced = solution[target_integral]
reduced

[[(1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0), 1414433104],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, 0), 2018052097],
 [(1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0), -616149827],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0), -1296074264],
 [(1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0), 1510467503],
 [(1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0), -1735337468],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0), 295686666],
 [(1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0), -137446141],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, 0, 0), 2053560367],
 [(1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0), -111299193],
 [(1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0), -43690813],
 [(1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0), 518442628],
 [(1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0), 907200528],
 [(1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0), -1532029951],
 [(1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0), -1227959168],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1), 581987461],
 [(1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0), -1238811958],
 [(1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0), 111571782],
 [(1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0), 1581581875],
 [(1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0), 1900

In [27]:
# The number of terms is the same as the number of masters, so the reduction is complete
len(reduced)

27